## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field

In [2]:
load_dotenv(override=True)

True

In [3]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if  openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key not set
Google API Key exists and begins AQ
OpenRouter API Key not set (and this is optional)
Groq API Key exists and begins gsk_


In [4]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

### It's easy to use any models with OpenAI compatible endpoints in 3 steps:

STEP 1: Find the OpenAI compatible base URL (see Guide 9 in the guides folder)

In [23]:
from agents import set_tracing_disabled

set_tracing_disabled(True)  # no OpenAI key, so don't upload traces to platform.openai.com

In [5]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

STEP 2: Create a python client library instance (the async version)

In [6]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
#openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

STEP 3: Create a model object

In [7]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
#kimi_model = OpenAIChatCompletionsModel(model="moonshotai/kimi-k2.6", openai_client=openrouter_client)
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

In [9]:
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")  # api_key is required by the client, ignored by Ollama
ollama_model = OpenAIChatCompletionsModel(model="qwen3:8b", openai_client=ollama_client)

sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
sales_agent2 = Agent(name="Ollama Sales Agent", instructions=instructions, model=ollama_model)
sales_agent3 = Agent(name="GPT-OSS Sales Agent",instructions=instructions, model=oss_model)

In [10]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [11]:
from messenger import send_email, push

USE_EMAIL = True

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [12]:
send_message("Yet another test", "Hooray!", "<html><body><h1>Hooray!</h1></body></html>")

In [13]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [14]:
tools = [tool1, tool2, tool3, send_email_tool]

In [17]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best funny sales email using the sales_agent tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=ollama_model)

In [18]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


✅ The email has been successfully sent to your prospects! 

This version strikes the best balance of **humor, clarity, and value proposition** while maintaining a professional tone. The playful subject line ("No More Sleepless Nights") and the "headache to win" metaphor add a light-hearted touch without undermining credibility.

Would you like to refine any part of the email further, or schedule follow-ups for the recipients?


## Check out the trace

https://platform.openai.com/traces

## Part 2: Structured Outputs

An LLM produces text in natural language. But we can have it instead produce a "python object".

This is accomplished using the usual trickery: clever prompts & json!

1. We specify a Python object  
2. In the System prompt, the LLM is instructed to respond in JSON and follow a Schema which represents the Python object  
3. The LLM outputs JSON, and the framework populates a Python object based on it

When we specify the Python object, we create a subclass of BaseModel, which is part of the Pydantic framework.

Pydantic is a framework that easily allows defining a JSON schema and mapping between Python and json.

NOTES:

1. There is something about the way this is done that IS really clever - if you're interested, look up "constrained decoding".
2. Not all providers support Structured Outputs.


In [19]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [20]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [21]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Ed
"""

In [35]:
checker = Agent(name="Checker", instructions="You review potential sales emails", model=gemini_model, output_type=EmailReview)
#result = await Runner.run(checker, email)

In [24]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=5, contains_placeholders=True)

In [25]:
review.is_professional

False

## Part 3: Guardrails

Guardrails are extremely important in AgenticAI. Put simply, they are controls that you code either in logic or with another LLM call, to prevent undesirable behavior.

For me, the Guardrails impementation in OpenAI Agents SDK feels a bit like "framework voodoo". I suspect their motivation was to show framework-level controls to address this important topic.

But it's simple and clean to implement guardrails explicitly, as separate Runner.run() calls, or checks in your tool implementations.

Regardless - let's take a look at the framework tooling.

https://openai.github.io/openai-agents-python/guardrails/


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Watch out for a Gotcha</h2>
            <span style="color:#ff7800;">There are 3 types of Guardrails in OpenAI Agents SDK: input, output and tool. Input guardrails only run for the first input to the first Agent in Runner.run(). Output guardrails only run for the final output of the last agent. If you have guardrails on other agents, they will never be called.
            </span>
        </td>
    </tr>
</table>

In [26]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem)

In [27]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=gemini_model, output_guardrails=[email_guardrail])

In [30]:

from agents import OutputGuardrailTripwireTriggered

try:
    result = await Runner.run(sales_agent1, "Write a cold sales email")
    result.final_output
except OutputGuardrailTripwireTriggered:
    print("Guardrail triggered - email is not professional")


Check out the trace:

https://platform.openai.com/traces

## On the other hand..

To state the obvious, this is simpler and will work in any framework

In [44]:
simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=gemini_model)
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)


Howdy partner. 

Listen, I know your inbox is lookin’ like a stampede of unwanted junk mail, so I’ll keep this shorter than a rattlesnake’s shadow. 

At ComplAI, we help folks like you wrangle their messy compliance operations so you don't end up havin' to face the regulatory sheriff with your pants down. 

Are you fixin' to save some time and keep your company safe, or are you happy keepin' on like a horse with a pebble in its shoe?

If you’ve got a moment, let’s hop on a quick call. If not, no hard feelin's—I’ll just ride off into the sunset.

Best,

[Your Name]
ComplAI Sales Manager


In [45]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional:
    print("The email is not professional")
else:
    print("Email is good")

The email is not professional


## Check out the trace:

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>

## OPTIONAL EXTRA: Sandbox Agents

This example will only work on Windows + WSL2, or Mac, or Linux

https://openai.github.io/openai-agents-python/sandbox_agents/

This is an execution harness - a runtime - "a persistent workspace where it can search large document sets, edit files, run commands, generate artifacts, and pick work back up from saved sandbox state."

You have to set up:
1. Manifest: the workspace
2. Capabilities: what it can do
3. SandboxRunConfig: where it runs

In [57]:
from pathlib import Path
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities, Shell
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

In [47]:
CODE_DIR = Path("code").resolve()
OUTPUT_DIR = Path("output").resolve()
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

In [48]:
CODE_DIR

PosixPath('/home/denisrepkov/projects/agents-udemy/2_openai/code')

In [49]:
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory:
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [58]:
manifest = Manifest(entries={"code": LocalDir(src=CODE_DIR)}, extra_path_grants=[SandboxPathGrant(path=str(OUTPUT_DIR))])
# Capabilities.default() = [Filesystem(), Shell(), Compaction()].
# Filesystem's apply_patch is a freeform (custom) tool and Compaction sends `context_management`;
# both only work with the Responses API. With OpenAIChatCompletionsModel (Groq/Gemini/Ollama) keep just Shell,
# which exposes plain function tools - the agent can read/write files with shell commands.
capabilities = [Shell()]
capabilities

[Shell(type='shell', session=None, run_as=None, configure_tools=None)]

In [59]:
run_config = RunConfig(sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()), workflow_name="Sandbox coding example")

In [62]:
agent = SandboxAgent(name="Engineer", instructions=instructions, model=ollama_model, default_manifest=manifest, capabilities=capabilities)

In [63]:
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

The file `/tmp/sandbox-local-3s11cwmt/code/orders.py` was processed successfully. Would you like me to:  
- Review its contents for issues?  
- Run tests or check for errors?  
- Modify specific parts of the file?  

Let me know how I can assist!


## OPTIONAL EXTRA: MCP Teaser!

In [64]:
from agents.mcp import MCPServerStreamableHttp


In [65]:
task = """
In the new SandboxAgents feature in the OpenAI Agents SDK as of May 2026, what is the role of the Manifest object?
Always be accurate. If you don't know the answer, say so.
"""

In [66]:
agent = Agent(name="Expert", instructions="Answer the question", model=ollama_model)
result = await Runner.run(agent, task)
print(result.final_output)

The role of the **Manifest object** in the **SandboxAgents** feature of the OpenAI Agents SDK (as of May 2026) is to **define and configure the sandboxed environment** in which agents operate. It typically includes metadata such as:

1. **Agent configurations**: Parameters, behaviors, or constraints for the agents.
2. **Environment settings**: Resources, tools, or dependencies available within the sandbox.
3. **Security/permission rules**: Restrictions on what actions agents can perform.
4. **Versioning**: Information about the sandbox environment or agent versions.

However, **without direct documentation from OpenAI** about this specific feature, this answer is based on standard practices for "Manifest" objects in SDKs and sandboxed systems. If OpenAI's official documentation provides a different definition, that would take precedence. Always refer to the official SDK documentation for precise details.


In [69]:
params = {"url": "https://mcp.context7.com/mcp", "timeout": 60}


async with MCPServerStreamableHttp(name="Context7", params=params) as server:
    agent = Agent(name="Expert", instructions="Use Context7 to answer the question", mcp_servers=[server], model=gemini_model)
    result = await Runner.run(agent, task)

print(result.final_output)

In the OpenAI Agents SDK, the **`Manifest`** object serves as the declarative "workspace contract" that defines the initial environment and contents for a fresh sandbox session.

As of May 2026, its primary roles include:

*   **Workspace Initialization:** It describes the layout and starting contents of a fresh sandbox workspace, ensuring the environment is prepared before an agent begins its tasks.
*   **Asset Management:** It allows you to define files and directories, copy local files, clone Git repositories, and attach remote storage mounts (such as S3, GCS, R2, Azure Blob, or Box).
*   **Environment Configuration:** It is used to set environment variables and define specific users or groups within the sandbox.
*   **Security and Portability:** Manifest entry paths are workspace-relative. They cannot be absolute paths or use `..` to escape the workspace, which ensures that the workspace contract remains portable across different clients (local, Docker, or hosted).

**Important Con

And see the traces:

https://platform.openai.com/traces
